In [ ]:
import torch
print(torch.__version__)
!nvcc --version
!python --version

In [ ]:
!git clone -b main https://github.com/okcze/GFlowNet-CombOpt.git

In [ ]:
%%capture
!pip install hydra-core omegaconf submitit hydra-submitit-launcher einops ipdb
if torch.cuda.is_available():
    # !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
    !pip install --pre dgl -f https://data.dgl.ai/wheels-test/torch-2.4/cu121/repo.html
    # !pip install  --force-reinstall dglgo -f https://data.dgl.ai/wheels-test/repo.html
else:
    !pip install dgl==1.1.3
!pip install --force-reinstall -v "networkx==2.8.8"
# !pip install --upgrade hydra-core

In [ ]:
# Imports after restart
import os
import torch
import numpy as np
import datetime
import shutil
import zipfile
from google.colab import drive
import itertools
import dgl
import hydra

In [ ]:
print(torch.__version__)
print(dgl.__version__)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd /content/GFlowNet-CombOpt

In [ ]:
%%capture
! python ./data/rbgraph_generator.py --num_graph 10 --graph_type small --save_dir ./data/rb200-300/train
# ! python ./data/rbgraph_generator.py --num_graph 50 --graph_type small --save_dir ./data/rb200-300/test

In [ ]:
with zipfile.ZipFile('/content/drive/MyDrive/master_data/test_500_mis.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/test_500_mis')

In [ ]:
os.makedirs('/content/GFlowNet-CombOpt/data/rb200-300/', exist_ok=True)
! cp -r /content/test_500_mis/content/GFlowNet-CombOpt/data/rb200-300/test /content/GFlowNet-CombOpt/data/rb200-300/test

In [ ]:
# Create directories for sampling outputs
!mkdir ./states
!mkdir ./logits

In [ ]:
### Set the config
task = 'mis'
ref_alg = 'mis_greedy'
dates = ['2025-03-30']
NUM_REPEAT = 1

param_set = [
    'frac_0_00_boost_1',
    'frac_0_05_boost_50',
    'frac_0_10_boost_50',
    'frac_0_25_boost_50',
    'frac_0_50_boost_50',
]

In [ ]:
# Fetch trained models
for date in dates:
    for param in param_set:
        if os.path.isdir(f'/content/drive/MyDrive/master_data/{date}/{task}/{param}'):
            !cp -r /content/drive/MyDrive/master_data/{date}/{task}/{param}/alg.pt /content/{param}.pt
            # !cp -r /content/drive/MyDrive/master_data/{date}/{task}/{param}/finetune/{param}/alg.pt /content/{param}.pt

In [ ]:
# Greedy algorithm inference
if torch.cuda.is_available():
    ! python -m gflownet.sample task={task} ref_alg={ref_alg} epochs=1 trainsize=10
else:
    ! python -m gflownet.sample task={task} d=-1 ref_alg={ref_alg} epochs=1 trainsize=10

In [ ]:
# Features GFN inference
for param in param_set:
    if torch.cuda.is_available():
        ! python -m gflownet.sample_features task={task} ref_alg={"mis_greedy_features"} epochs=1 trainsize=10 alg_load_path=/content/{param}.pt
    else:
        ! python -m gflownet.sample_features task={task} d=-1 ref_alg={"mis_greedy_features"} epochs=1 trainsize=10 alg_load_path=/content/{param}.pt

In [ ]:
# # GFN versions inference
# for param in param_set:
#     if torch.cuda.is_available():
#         ! python -m gflownet.sample task={task} epochs={NUM_REPEAT} trainsize=10 testsize=50 alg_load_path=/content/{param}.pt
#     else:
#         ! python -m gflownet.sample d=-1 task={task} epochs={NUM_REPEAT} trainsize=10 testsize=50 alg_load_path=/content/{param}.pt

In [ ]:
# Logits
if torch.cuda.is_available():
    ! python -m gflownet.logits task={task} epochs=1 trainsize=10 alg_load_path=/content/frac_0_00_boost_1.pt
else:
    ! python -m gflownet.logits d=-1 task={task} epochs=1 trainsize=10 alg_load_path=/content/frac_0_00_boost_1.pt

In [ ]:
class MetricsScorer:
    def __init__(self, task):
        self.task = task

    def score_traj(self, traj):
        s = traj[-1]
        if self.task == 'mds':
            return -(s == 1).sum()
        elif self.task == 'mis':
            return np.sum(s)
        elif self.task == 'maxcut':
            return np.sum(s)
        else:
            raise NotImplementedError

    def dropdown(self, baseline_score, ref_score):
        """Measures relative loss comparing to vanilla GFN solution"""
        return (ref_score - baseline_score)/np.abs(baseline_score)

    def likelihood(self, logits, states):
        """Measures likelihood of arbitrary trajectory under vanilla GFN policy"""
        lh = 0  # Start with 0 since we're summing log probabilities
        for i in range(len(logits)-1):
            differing_indices = np.where(states[i] != states[i+1])[0]
            for idx in differing_indices:
                if states[i+1][idx] == 1:
                    lh += logits[i][idx]  # Sum the log probabilities (logits)
        return lh

    def intersection(self, t_greedy, t2):
        """Measures intersection between two solutions"""
        s1, s2 = t_greedy[-1], t2[-1]
        s1_nodes, s2_nodes = np.where(s1 == 1)[0], np.where(s2 == 1)[0]
        return len(set(s1_nodes) & set(s2_nodes))/len(s1_nodes)

    def get_action(self, s, s_next):
        curr = set(np.where(s==1)[0].tolist())
        next = set(np.where(s_next==1)[0].tolist())
        return next.difference(curr)

    def similarity(self, t1, t2):
        """Measures similarity between two trajectories"""
        min_len = min(len(t1), len(t2))
        max_len = max(len(t1), len(t2))
        sim = []
        for step in range(min_len-1):
            a1, a2 = self.get_action(t1[step], t1[step+1]), self.get_action(t2[step], t2[step+1])
            sim.append(a1 == a2)
        for step in range(min_len, max_len):
            sim.append(False)
        return sim

    def get_max_logit_action(self, logits):
        """Finds the logits maximized actions"""
        max_trajectory = []
        for i in range(len(logits)-1):
            maximized_action = int(np.argmax(logits[i]))
            max_trajectory.append(maximized_action)
        return max_trajectory

    def actions_traj(self, t):
        """Extracts actions from trajectory"""
        t_len = len(t)
        actions = []
        for step in range(t_len-1):
            a = self.get_action(t[step], t[step+1])
            if a:
                actions.append(a.pop())
            else:
                actions.append(None)
        return actions

In [ ]:
# Likelihoods and Scores
scorer = MetricsScorer(task)
algorithms = os.listdir('/content/GFlowNet-CombOpt/states')

lhs_scores = []
for it in range(NUM_REPEAT):

    results_to_gfn = {}
    for alg in algorithms:

        scores = []
        lhs = []
        max_actions = []
        sampling_actions = []
        nodes_features = []

        states = os.listdir(f'/content/GFlowNet-CombOpt/states/{alg}/{it}')
        states = [s for s in states if "features" not in s]
        for graph in states:

            gfn_traj = np.load(f'./states/frac_0_00_boost_1/{it}/{graph}')
            ref_traj = np.load(f'./states/{alg}/{it}/{graph}')
            if alg != 'MISGreedy':
                ref_traj_features = np.load(f'./states/{alg}/{it}/{graph.replace(".npy", "")}_features.npy')
            else:
                ref_traj_features = []
            logits = np.load(f'./logits/{alg}/{graph}')
            logits = logits.squeeze(axis=1)

            lhs.append(scorer.likelihood(logits, ref_traj))
            scores.append(scorer.score_traj(ref_traj))
            max_actions.append(scorer.get_max_logit_action(logits))
            sampling_actions.append(scorer.actions_traj(ref_traj))
            nodes_features.append(ref_traj_features)

        scores = np.array(scores)
        lhs = np.array(lhs)

        results_to_gfn[alg] = {
            "likelihood": lhs,
            "score": scores,
            "max_actions": max_actions,
            "sampling_actions": sampling_actions,
            "nodes_features": nodes_features,
        }
    lhs_scores.append(results_to_gfn)

In [ ]:
all_data = []

for alg in algorithms:
    for it in range(NUM_REPEAT):
        for x in lhs_scores[it][alg]["score"]:
            all_data.append({"Algorithm": alg, "Score": x, "Iteration": it})
        for x in lhs_scores[it][alg]["likelihood"]:
            all_data.append({"Algorithm": alg, "Likelihood": x, "Iteration": it})

df = pd.DataFrame(all_data)

In [ ]:
# Compare to the reference algorithm
scorer = MetricsScorer(task)
algorithms = os.listdir('/content/GFlowNet-CombOpt/states')

reg_results = []
for it in range(NUM_REPEAT):

    results_iter = {}
    for alg in algorithms:

        similarities = []
        intersections = []
        scores = []
        graph_ids = []

        states = os.listdir(f'/content/GFlowNet-CombOpt/states/{alg}/0')
        states = [s for s in states if "features" not in s]
        for graph_id, graph in enumerate(states):

            greedy_traj = np.load(f'./states/MISGreedy/0/{graph}')
            ref_traj = np.load(f'./states/{alg}/{it}/{graph}')

            similarities.append(scorer.similarity(greedy_traj, ref_traj))
            intersections.append(scorer.intersection(greedy_traj, ref_traj))
            scores.append(scorer.score_traj(ref_traj))
            graph_ids.append(graph_id)


        results_iter[alg] = {
            "similarity": similarities,
            "intersection": intersections,
            "score": scores,
            "graph_id": graph_ids,
        }
    reg_results.append(results_iter)

In [ ]:
reg_data = []

for alg in algorithms:
    for it in range(NUM_REPEAT):
        for x in reg_results[it][alg]["similarity"]:
            reg_data.append({"Algorithm": alg, "Similarity": x, "Iteration": it})
        for x in reg_results[it][alg]["intersection"]:
            reg_data.append({"Algorithm": alg, "Intersection": x, "Iteration": it})
        for x in reg_results[it][alg]["score"]:
            reg_data.append({"Algorithm": alg, "Score": x, "Iteration": it})
        for x in reg_results[it][alg]["graph_id"]:
            reg_data.append({"Algorithm": alg, "Graph ID": x, "Iteration": it})

df_reg = pd.DataFrame(reg_data)

In [ ]:
# Rename frac_0_00_boost_1 to canonical GFN
df.loc[df.Algorithm == 'frac_0_00_boost_1', 'Algorithm'] = 'Canonical GFN'
df_reg.loc[df_reg.Algorithm == 'frac_0_00_boost_1', 'Algorithm'] = 'Canonical GFN'
# Rename frac_0_50_boost_50 to 'f=0.5 boost=50'
# df.loc[df.Algorithm == 'frac_0_50_boost_50', 'Algorithm'] = 'Regularized GFN'
# df_reg.loc[df_reg.Algorithm == 'frac_0_50_boost_50', 'Algorithm'] = 'Regularized GFN'

In [ ]:
# algorithms = ["Canonical GFN", "Regularized GFN", "MISGreedy"]
# hue_order = algorithms

## Interesting Scores
frac_0_10

In [ ]:
df_score_comp = pd.concat([df[df.Algorithm == "Canonical GFN"].dropna(subset='Score').reset_index(), df[df.Algorithm == "frac_0_10_boost_50"].dropna(subset='Score').reset_index()], axis=1)
df_score_comp['score_diff'] = df_score_comp.iloc[:, -3] - df_score_comp.iloc[:, -8]
df_score_comp.sort_values("score_diff", ascending=False).head(15)

In [ ]:
df_score_comp = pd.concat([df[df.Algorithm == "Canonical GFN"].dropna(subset='Score').reset_index(), df[df.Algorithm == "frac_0_25_boost_50"].dropna(subset='Score').reset_index()], axis=1)
df_score_comp['score_diff'] = df_score_comp.iloc[:, -3] - df_score_comp.iloc[:, -8]
df_score_comp.sort_values("score_diff", ascending=False).head(15)

In [ ]:
df_reg[df_reg.Algorithm == "frac_0_50_boost_50"].dropna(subset='Intersection').reset_index().sort_values("Intersection", ascending=False).head(15)

Examples

In [ ]:
def sampled_in_set(df):
    count = 0
    for row in df.iterrows():
        sampled_action = row[1]['Sampled actions']
        min_values = np.where(row[1]['Features values'] == np.min(row[1]['Features values']))
        allowed_actions = np.array(row[1]['Features nodes'][min_values])
        if sampled_action in allowed_actions:
            count += 1
    return count

def evaluate_solution(df):
    df = df.dropna()
    solution_score = df['Sampled actions'].notna().sum()
    on_proxy = sampled_in_set(df)
    return solution_score, on_proxy

In [ ]:
presented_columns = ['Step', 'Sampled Node Id', 'Sampled Node Degree', 'Greedy Node Id', 'Minimal Node Degree']

def compare_solutions(graph_id, reg_model):
    solution_scores = []
    on_proxy_scores = []
    intersection_with_vanilla = []
    solutions = {}
    for model in param_set:
        features_nodes= lhs_scores[0][model]['nodes_features'][graph_id][::2],
        features_values= lhs_scores[0][model]['nodes_features'][graph_id][1::2],
        df_actions = pd.DataFrame({
            "Max actions": lhs_scores[0][model]['max_actions'][graph_id],
            "Sampled actions": lhs_scores[0][model]['sampling_actions'][graph_id],
            "Features nodes": list(features_nodes[0]),
            "Features values": list(features_values[0]),
        })
        solution_score, on_proxy = evaluate_solution(df_actions)
        solution_scores.append(solution_score)
        on_proxy_scores.append(on_proxy)
        solutions[model] = set(df_actions['Sampled actions'].dropna().values)

    for model in param_set:
        intersection_with_vanilla.append(len(solutions['frac_0_00_boost_1'].intersection(solutions[model])))

    df_proxy = pd.DataFrame({"GFN version": param_set, "Solution score": solution_scores, "Actions on proxy": on_proxy_scores, "Actions on vanilla GFN": intersection_with_vanilla})
    df_proxy.to_latex(
        f'/content/{graph_id}_summary.tex',
        float_format="%.2f",
        caption=f"Summary of GFN versions on graph {graph_id}.",
        label=f"tab:{graph_id}_summary",
        column_format="l|ccc",
        bold_rows=True,
        index=False
    )

    features_nodes= lhs_scores[0][reg_model]['nodes_features'][graph_id][::2],
    features_values= lhs_scores[0][reg_model]['nodes_features'][graph_id][1::2],
    df_actions_reg = pd.DataFrame({
        "Step": range(1, len(lhs_scores[0][reg_model]['max_actions'][graph_id])+1),
        "Sampled action": lhs_scores[0][reg_model]['sampling_actions'][graph_id],
        "Node degree": [deg[0] for deg in features_values[0]],
        "Nodes IDs": list(features_nodes[0]),
        "Nodes degrees": list(features_values[0]),
    })
    df_actions_reg = df_actions_reg.dropna()
    df_actions_reg['Sampled Node Id'] = df_actions_reg['Sampled action'].astype(int)
    df_actions_reg['Sampled Node Degree'] = df_actions_reg.apply(lambda row: row['Nodes degrees'][row['Sampled Node Id'] == row['Nodes IDs']][0], axis=1)
    df_actions_reg['Minimal Node Degree'] = df_actions_reg.apply(lambda row: row['Nodes degrees'][0], axis=1)
    df_actions_reg['Greedy Node Id'] = df_actions_reg.apply(lambda row: row['Nodes IDs'][0], axis=1)
    df_actions_reg.to_latex(
        f'/content/{graph_id}_reg.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.5, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cll",
        bold_rows=True,
        index=False
    )

    model = "frac_0_00_boost_1"
    features_nodes= lhs_scores[0][model]['nodes_features'][graph_id][::2],
    features_values= lhs_scores[0][model]['nodes_features'][graph_id][1::2],
    df_actions_vanilla = pd.DataFrame({
        "Step": range(1, len(lhs_scores[0][model]['max_actions'][graph_id])+1),
        "Sampled action": lhs_scores[0][model]['sampling_actions'][graph_id],
        "Node degree": [deg[0] for deg in features_values[0]],
        "Nodes IDs": list(features_nodes[0]),
        "Nodes degrees": list(features_values[0]),
    })
    df_actions_vanilla = df_actions_vanilla.dropna()
    df_actions_vanilla['Sampled Node Id'] = df_actions_vanilla['Sampled action'].astype(int)
    df_actions_vanilla['Sampled Node Degree'] = df_actions_vanilla.apply(lambda row: row['Nodes degrees'][row['Sampled Node Id'] == row['Nodes IDs']][0], axis=1)
    df_actions_vanilla['Minimal Node Degree'] = df_actions_vanilla.apply(lambda row: row['Nodes degrees'][0], axis=1)
    df_actions_vanilla['Greedy Node Id'] = df_actions_vanilla.apply(lambda row: row['Nodes IDs'][0], axis=1)
    df_actions_vanilla.to_latex(
        f'/content/{graph_id}_vanilla.tex',
        float_format="%.2f",
        caption=f"Vanilla GFN (f=0.0, b=1) on graph {graph_id}.",
        label=f"tab:{graph_id}_vanilla",
        column_format="l|cll",
        bold_rows=True,
        index=False
    )

    return df_proxy, df_actions_reg, df_actions_vanilla

In [ ]:
# 3
graph_id = 274
df_proxy, df_actions_reg, df_actions_vanilla = compare_solutions(274, "frac_0_10_boost_50")

In [ ]:
df_tex = df_actions_reg[presented_columns]
df_tex.to_latex(
        f'/content/patterns_3.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.1, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

In [ ]:
# 4
graph_id = 392
df_proxy, df_actions_reg, df_actions_vanilla = compare_solutions(graph_id, "frac_0_10_boost_50")

In [ ]:
df_tex = df_actions_reg[presented_columns]
df_tex.to_latex(
        f'/content/patterns_4.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.1, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

In [ ]:
# 6
graph_id = 184
df_proxy, df_actions_reg, df_actions_vanilla = compare_solutions(graph_id, "frac_0_10_boost_50")

In [ ]:
df_proxy

In [ ]:
df_tex = df_actions_reg[presented_columns]
df_tex.to_latex(
        f'/content/patterns_6.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.1, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

Poor regularization

In [ ]:
graph_id = 129
model = "frac_0_50_boost_50"
df_proxy, df_actions_reg, df_actions_vanilla = compare_solutions(graph_id, model)

In [ ]:
df_tex = df_actions_reg[['Step', 'Sampled Node Id', 'Sampled Node Degree', 'Greedy Node Id', 'Minimal Node Degree']]
df_tex.to_latex(
        f'/content/weak_reg.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.5, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

In [ ]:
df_tex = df_actions_vanilla[['Step', 'Sampled Node Id', 'Sampled Node Degree', 'Greedy Node Id', 'Minimal Node Degree']]
df_tex.to_latex(
        f'/content/weak_reg_vanilla.tex',
        float_format="%.2f",
        caption=f"Vanilla GFN on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

Strong regularization

In [ ]:
graph_id = 271
model = "frac_0_50_boost_50"
df_proxy, df_actions_reg, df_actions_vanilla = compare_solutions(graph_id, model)

In [ ]:
df_tex = df_actions_reg[['Step', 'Sampled Node Id', 'Sampled Node Degree', 'Greedy Node Id', 'Minimal Node Degree']]
df_tex.to_latex(
        f'/content/strong_reg.tex',
        float_format="%.2f",
        caption=f"Regularized GFN (f=0.5, b=50) on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex

In [ ]:
df_tex = df_actions_vanilla[['Step', 'Sampled Node Id', 'Sampled Node Degree', 'Greedy Node Id', 'Minimal Node Degree']]
df_tex.to_latex(
        f'/content/strong_reg_vanilla.tex',
        float_format="%.2f",
        caption=f"Vanilla GFN on graph {graph_id}.",
        label=f"tab:{graph_id}_reg",
        column_format="l|cc|cc",
        bold_rows=True,
        index=False
)
df_tex